# About

- Goal
  - 매우 긴 conversation history를 위한 agent memory 기법들 이론 정리
  - 대표적인 benchmark인 `LongMemEval`로 연습 -> 이론 공부를 넘어 implementation details 확인 및 정리
  - 각 방식의 장단점을 실제 데이터로 확인, 실무 case를 위한 insight 얻기
  - Future study 방향 선정
- Period: 2026-11-02 ~ 2026-11-04
- Member: aruwad

# Summary

```markdown
1. No Memory
  - 방법: 그냥 메모리 없이 채팅
  - 장점: generation이 빠르다, context length 넉넉하다 등
  - 한계: 당연히 과거 채팅 기억 x

2. Chat History
  - 방법: Last history를 일정 길이만큼 prepend, context length 도달시 오래된 history부터 삭제
  - 장점: 구현이 단순하다
  - 한계
    - 금방 context length 포화
    - 중요한 정보가 조금만 지나도 사라짐

3. Summarization
  - 방법: current context length가 일정 이상일 때마다, 별도의 summarizer가 context를 summarize하여 압축
  - 장점: 오래된 context를 어느정도 보존 가능
  - 한계
    - Summarization 호출 횟수에 따라 LLM call 급증할 수 있음
    - 결국 context size를 차지하므로, working memory 정도면 모를까, 대용량의 knowledge 등을 표현하기 어려움
    - 어떤 것을 저장/retrieval 할 지에 대한 기준이 없음

4. History RAG
  - 방법: history를 chunk 단위로 split/embedding 후 vector DB에 저장, hybrid retrieval (dense/BM25) + reranking
  - 장점: 대용량 history 보존 가능, retrieval에 별도의 LLM call이 없어 빠른 실행
  - 한계
    - Memory representation이 여전히 raw text (chunk)에 머물러, working/semantic/procedure/episodic memory 등 표현 어려움
    - 정교한 update/forgetting/consolidation 알고리즘이 없어, 단순히 추가만 할 경우 vector DB가 무제한 늘어남

5. Structured Memory
  - 방법: advanced memory representation
    - Working memory: 현재 작업/상황에 대한 context로서 일시적으로 highly attended
    - Semantic memory: user profile, fact
    - Episodic memory: who/when/where/whom/what/how, etc.
    - Procedure memory: structured process for specific task
  - 장점: 단순 raw text보다 LLM이 이해하기 쉬운 형태로 기억 생성/업데이트/삭제/참조 가능 -> 보다 나은 retrieval 및 explainability 가능
  - 한계
    - 어떤 raw text를 기억해야 할 지에 대한 기준이 없는데, Raw text -> structured memory 생성에 LLM call이 필요하므로, 모든 raw text에 대해 호출했다간 실행시간 급증할 수 있음
    - Memory 간의 semantic relationship을 고려하지 못해, associative memory 등에 활용할 수 없음
    - Task가 요구하는 정보를 제대로 structuring하지 않으면 제대로 기억 x (예: 공연 관람 사건만 저장하고 작품명·날짜를 누락하면 구체적인 회상 질문에 답하기 어려움)

6. Graph RAG
  - 방법: history를 단순 chunk가 아닌 semantically represented as node & semantic relationship as edge
  - 장점: entity 간 관계를 따라 관련 사건/사실을 함께 탐색하고, 관계와 원문 근거를 같이 확인 가능
  - 한계
    - 관계 추출에 LLM call이 필요해 indexing이 느리고, 추출 과정에서 제목/수치/추천/시간 정보가 누락될 수 있음
    - 관계가 검색되어도 원문 발췌를 과도하게 제한하면 정답 근거가 사라짐
    - 연결된 관계만으로는 화자 구분, 최신 정보 선택, 여러 사실의 종합이 보장되지 않음
  - 확인한 문제 및 수정
    - 원문 청크당 발췌 2개 제한이 정답을 버리는 사례 확인 -> 전체 발췌 예산 안에서 reranking
    - 그래프 관계에 없는 사실 -> 기존 원문 embedding 검색 + 세부 발췌 BM25로 보완
    - 청크 경계에서 빠진 화자/날짜 -> 이전 청크의 실제 header에서 복원
    - embedding의 비동기 경로에서 배치가 풀리는 문제 -> 동기 배치 호출로 복구
    - 답변은 qwen3:8b, 관계 추출은 qwen3:4b-instruct 사용; 모델/공통 답변 prompt는 유지

7. 평가
  - Benchmark: LongMemEval-S-cleaned
    - 여러 과거 대화에서 개인 정보/사건/추천 등을 회상하는 500문항; 단일 세션(user/assistant/preference), multi-session, temporal reasoning, knowledge update 및 abstention 평가
    - 500문항 전체의 데이터 구조 점검: 세션 순서가 항상 시간순은 아니며, has_answer 표시만으로 정답 세션을 빠짐없이 선택할 수 없음
    - 질문 시각보다 늦은 같은 날짜 세션에도 정답이 있어 단순 시각 cutoff를 적용하면 근거 손실 가능
    - 질문 유형/정답/정답 세션 표시는 평가와 분석에만 사용; 검색/답변에 전달하지 않음
  - LLM-as-a-judge (local LLM)
    - 정답과 생성 답변의 의미 일치 여부를 유형별 rubric으로 평가; abstention도 별도 확인
    - 강제 JSON Schema에서 추론과 boolean 출력이 모순되는 사례 확인 -> 일반 JSON 요청 + 엄격한 파싱, 대조군 실패 시 평가 중단
    - 수정 후에도 local judge의 의미 판단 오류가 남아, 최종 점수는 방법명을 가린 두 독립 검토로 확인; 주요 비교의 모든 판정 일치
  - 비교 범위 및 결과
    - 7개 유효 유형별 4문항, seed 42로 총 28문항 선정: 개발 14문항 + 별도 검증 14문항
    - 개발: History RAG 12/14, 수정 Graph RAG 13/14
    - 검증: History RAG 6/14, 수정 Graph RAG 7/14
    - 합계: History RAG 18/28 (64.3%), 수정 Graph RAG 20/28 (71.4%)
    - 전체 500문항 성능이나 통계적 우위를 입증한 결과는 아님; 두 방식의 검색 문맥량도 다름
    - 캐시 재사용/병렬 실행으로 전체 실행시간 단축률은 확정하지 않음
    - 최종 결과: experiments/graph_rag_audit_20261104/report/; 원본 실험 기록에는 제외된 변형도 포함

8. Concluding remarks
  - 이 표본에서는 수정 Graph RAG가 History RAG보다 높은 정답률을 보였으나, 그래프 사용 자체의 일반적인 우위로 해석할 수 없음
  - 모델 크기뿐 아니라 정보 보존 -> 검색 -> 근거 해석 -> 채점의 각 단계에서 실패 원인을 분리해야 함
  - 원문 근거를 보존하는 hybrid retrieval이 관계 추출의 정보 손실을 보완; Graph RAG에서도 원문이 중요
  - 남은 Graph RAG 오답 8개: 근거 누락 3개, 근거가 있어도 해석 실패 4개, 혼합 1개
  - 다음 검증: 전체 500문항, 동일 문맥 예산, 신뢰할 수 있는 채점, cold indexing/query 시간을 분리한 비교

9. Future Study
  - HeLa-Mem
    - https://arxiv.org/abs/2604.16839
    - Hebbian learning for associative memory
    - Episodic memory graph와 semantic memory를 통합

```

# Setup

In [1]:
import json
import random
import re
from concurrent.futures import ThreadPoolExecutor
from functools import lru_cache
from pathlib import Path
from pprint import pprint
from time import perf_counter
from uuid import uuid4

import numpy as np
import pandas as pd
from dotenv import dotenv_values
from huggingface_hub import hf_hub_download
from IPython.display import display
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage, SystemMessage, convert_to_messages
from langchain_core.messages.utils import count_tokens_approximately
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.runtime import Runtime
from langgraph.store.sqlite import SqliteStore
from langmem import create_memory_store_manager, create_search_memory_tool
from llama_index.core import Document as GraphDocument, PropertyGraphIndex
from llama_index.core.graph_stores.types import ChunkNode
from llama_index.core.indices.property_graph import (
    SimpleLLMPathExtractor,
    VectorContextRetriever,
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.langchain import LangchainEmbedding
from llama_index.llms.ollama import Ollama
from pydantic import BaseModel, Field
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
from tqdm.auto import tqdm
from transformers import AutoTokenizer


cfg = dotenv_values()
seed = int(cfg['SEED'])
print(f'Seed: {seed}')

c:\Users\yana\Desktop\Library\AI\ai-topic\ai-topic-memory\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1105 02:31:00.217000 84512 Lib\site-packages\torch\utils\_pytree.py:646] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1105 02:31:00.314000 84512 Lib\site-packages\torch\utils\_pytree.py:646] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Seed: 42


# Chat Model

In [2]:
chat = ChatOllama(
    model="qwen3:8b",
    reasoning=True,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    num_ctx=32_768,
    num_predict=2_048,
    seed=seed,
)


# System prompt
system_prompt = SystemMessage(
    content=(
        "Answer the user's question briefly from the conversation. "
        "Use what the user stated or clearly implied. One relevant statement is sufficient. "
        "Match the person, event, and time asked about. "
        "If the conversation does not support an answer, say you do not know."
    )
)

# Data

- `xiaowu0162/longmemeval-cleaned`

In [3]:
path = hf_hub_download(
    repo_id="xiaowu0162/longmemeval-cleaned",
    repo_type="dataset",
    filename="longmemeval_s_cleaned.json",
)

with open(path, encoding="utf-8") as f:
    dataset = json.load(f)


# Sample
sample = dataset[0]

print(f"[question] {sample["question"]}\n")
print(f"[question_type] {sample["question_type"]}\n")
print(f"[answer] {sample["answer"]}\n")
print(f"[haystack_sessions]\n{sample["haystack_sessions"][-1][-1]}\n")     # list of conversation history

# Responses
responses = {}

[question] What degree did I graduate with?

[question_type] single-session-user

[answer] Business Administration

[haystack_sessions]
{'role': 'assistant', 'content': "Kitchen pantry organization is a crucial aspect of maintaining a functional and efficient kitchen. Here are some suggestions to help you maximize the storage space in your pantry:\n\n**Purge and Categorize:**\n\n1. **Purge expired or unused items**: Remove any expired, stale, or unused food items from your pantry to free up space and reduce clutter.\n2. **Categorize items**: Group similar items together, such as baking supplies, snacks, canned goods, and spices, to make it easier to find what you need.\n\n**Shelving and Storage:**\n\n1. **Install adjustable shelves**: Adjustable shelves allow you to customize the height and spacing of your shelves to accommodate different-sized items.\n2. **Use stackable containers**: Stackable containers, baskets, or bins can help maximize vertical storage space and keep items organiz

# Helpers

In [4]:
def dated_sessions(sample):
    sessions = zip(sample["haystack_dates"], sample["haystack_sessions"], strict=True)
    for date, session in sessions:
        yield [
            {"role": msg["role"], "content": f"[Conversation date: {date}] {msg['content']}"}
            for msg in session
        ]


def dated_question(sample):
    return HumanMessage(
        content=f"Question date: {sample['question_date']}\nQuestion: {sample['question']}"
    )

# 1. No Memory

In [5]:
def chat_no_memory(data):
    start = perf_counter()
    config = {"configurable": {"thread_id": "no_memory"}}
    responses = []

    for sample in data:
        question = sample["question"]

        messages = [
            system_prompt,
            HumanMessage(question),
        ]

        answer = chat.invoke(
            input=messages,
            config=config,
        )

        exec_time = round(perf_counter() - start, 4)
        responses.append(
            {
                "question": question,
                "answer": answer,
                "exec_time": exec_time,
            }
        )

    return responses

In [6]:
responses = chat_no_memory(dataset[:2])

print("# 1. No Memory")

for response in responses:
    print(f"- Question: {response['question']}")
    print(f"- Answer: {response['answer'].content}")
    print(f"- Execution time: {response['exec_time']}\n")

# 1. No Memory
- Question: What degree did I graduate with?
- Answer: You graduated with a Bachelor's degree in Computer Science.
- Execution time: 4.3348

- Question: How long is my daily commute to work?
- Answer: I don't know. You haven't provided any information about your daily commute duration.
- Execution time: 4.9999



# 2. Chat History

In [7]:
max_history_length = 1_000


def chat_with_history(data):
    start = perf_counter()

    agent = create_agent(
        model=chat,
        system_prompt=system_prompt,
    )

    responses = []

    for i, sample in enumerate(data):
        config = {"configurable": {"thread_id": f"chat_history_{i}"}}
        question = sample["question"]

        flatten = [msg for session in dated_sessions(sample) for msg in session]

        messages = []
        for message in reversed(flatten):
            if count_tokens_approximately([message, *messages]) < max_history_length:
                messages.insert(0, message)
            else:
                break

        answer = agent.invoke(
            input={
                "messages": [
                    *messages,
                    dated_question(sample),
                ]
            },
            config=config,
        )

        exec_time = round(perf_counter() - start, 4)

        responses.append(
            {
                "question": question,
                "answer": answer,
                "messages": messages,
                "exec_time": exec_time,
            }
        )

    return responses

In [8]:
responses = chat_with_history(dataset[:2])

print("# 2. Chat history")

for response in responses:
    print(f"- Question: {response['question']}")
    print(f"- Answer: {response['answer']['messages'][-1].content}")
    print(f"- Execution time: {response['exec_time']}")
    print(f"- Messages ({len(response['messages'])}):")
    for message in response['messages']:
        print(f"  - {message}")

    print("\n")

# 2. Chat history
- Question: What degree did I graduate with?
- Answer: I don't know.
- Execution time: 0.51
- Messages (2):
  - {'role': 'user', 'content': "[Conversation date: 2023/05/30 (Tue) 21:40] I'd like to get some more ideas on how to organize my kitchen pantry. Can you give me some suggestions on how to maximize the storage space in my pantry?"}
  - {'role': 'assistant', 'content': "[Conversation date: 2023/05/30 (Tue) 21:40] Kitchen pantry organization is a crucial aspect of maintaining a functional and efficient kitchen. Here are some suggestions to help you maximize the storage space in your pantry:\n\n**Purge and Categorize:**\n\n1. **Purge expired or unused items**: Remove any expired, stale, or unused food items from your pantry to free up space and reduce clutter.\n2. **Categorize items**: Group similar items together, such as baking supplies, snacks, canned goods, and spices, to make it easier to find what you need.\n\n**Shelving and Storage:**\n\n1. **Install adjust

# 3. Summarization

In [9]:
summary_prompt = """Update the running summary of conversations with the current user.
Preserve names, dates, events, preferences, and relevant information.
Keep event dates distinct from conversation dates, including dates of corrected facts.
Distinguish user facts from assistant suggestions. Apply explicit corrections.
Omit greetings and repetition. Do not invent information.
Return only a concise summary, at most 150 words.

<messages>
{messages}
</messages>"""


summarizer = SummarizationMiddleware(
    model=ChatOllama(
        model="qwen3:1.7b",             # slightly light model
        num_predict=384,                # short summary, with room for names and dates
        num_ctx=16_384,                 # room for the batch, prompt, and generated summary
        reasoning=False,
    ),
    trigger=("tokens", 6_000),
    keep=("tokens", 512),               # keep last 512 tokens as is
    token_counter=count_tokens_approximately,
    summary_prompt=summary_prompt,
    trim_tokens_to_summarize=None,      # summarize the full batch without dropping older messages
)

runtime = Runtime()


def chat_with_summary(data):
    agent = create_agent(model=chat, system_prompt=system_prompt)
    responses = []

    for i, sample in enumerate(data):
        start = perf_counter()
        question = sample["question"]
        messages = []

        for session in tqdm(
            dated_sessions(sample),
            total=len(sample["haystack_sessions"]),
            desc=f"Sample {i}",
        ):
            for message in convert_to_messages(session):
                # add & summarize one message at a time
                messages.append(message)

                result = summarizer.before_model({"messages": messages}, runtime)
                if result:
                    messages = result["messages"][1:]

        summary = (
            messages[0].content
            if messages and messages[0].additional_kwargs.get("lc_source") == "summarization"
            else None
        )

        answer = agent.invoke(
            {"messages": [*messages, dated_question(sample)]},
            config={"configurable": {"thread_id": f"chat_summary_{i}"}},
        )

        responses.append({
            "question": question,
            "answer": answer,
            "summary": summary,
            "exec_time": round(perf_counter() - start, 4),
        })

    return responses

# 4. History RAG

In [6]:
embedding_name = "BAAI/bge-small-en-v1.5"
embeddings = HuggingFaceEmbeddings(
    model_name=embedding_name,
    encode_kwargs={"normalize_embeddings": True},
)
splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    AutoTokenizer.from_pretrained(embedding_name), chunk_size=320, chunk_overlap=32
)
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")


def tokenize(text):
    return re.findall(r"\w+", text.lower())


@lru_cache(maxsize=4)
def history_index(turns):
    chunks = [
        f"[{date} | {role}] {part}"
        for date, role, content in turns
        for part in splitter.split_text(content)
    ]
    store = InMemoryVectorStore.from_texts(chunks, embedding=embeddings)
    bm25 = BM25Okapi([tokenize(chunk) for chunk in chunks])
    return chunks, store, bm25


def chat_with_history_rag(data):
    agent = create_agent(model=chat, system_prompt=system_prompt)
    responses = []

    for sample in tqdm(data, desc="History RAG"):
        start = perf_counter()
        question = sample["question"]

        turns = tuple(
            (date, msg["role"], msg["content"])
            for date, session in zip(
                sample["haystack_dates"], sample["haystack_sessions"], strict=True
            )
            for msg in session
        )
        chunks, store, bm25 = history_index(turns)
        dense = store.similarity_search(
            "Represent this sentence for searching relevant passages: " + question, k=5
        )
        sparse = bm25.get_top_n(tokenize(question), chunks, n=5)
        candidates = list(dict.fromkeys([doc.page_content for doc in dense] + sparse))

        ranked = reranker.rank(question, candidates, top_k=3)
        retrieved = [candidates[hit["corpus_id"]] for hit in ranked]

        history = "\n\n".join(retrieved)
        answer = agent.invoke({"messages": [HumanMessage(content=(
            f"Conversation excerpts:\n{history}\n\n"
            f"Question date: {sample['question_date']}\nQuestion: {question}"
        ))]})

        responses.append({
            "question": question,
            "answer": answer,
            "retrieved": retrieved,
            "exec_time": round(perf_counter() - start, 4),
        })

    return responses

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 5971.63it/s]


# 5. Structured Memory

- Working memory
  - "What am I doing now?"
- Semantic memory
  - "What do I know?"
  - Profile: a collection of components representing an entity that might be frequently updated, e.g. {'name': 'yana', 'current_study': 'nlp'}.
  - Collection: a collection of individual facts, e.g. {'text': 'gpt is the most popular LLM', 'topic': 'llm'}
- Episodic memory
  - "What event did I experience?"
- Procedure memory
  - "How do I perform this task?"
  - Tool: a fixed, well-defined task.
  - Procedure memory: an uncertain, changable procedure that needs an LLM reasoning.

In [11]:
class Episode(BaseModel):
    """One memorable event, fact, preference, or recommendation."""
    who: str = Field(description="Who performed the action.")
    action: str = Field(description="Short action or event.")
    whom: str = Field(description="Person or entity targeted by the action. Empty if none.")
    when: str = Field(description="When the event happened. Empty if unknown.")
    where: str = Field(description="Where the event happened. Empty if unknown.")


memory_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
)


def chat_with_structured_memory(dataset):
    db_path = Path("tmp/memory/structured_memory.sqlite").resolve()
    db_path.parent.mkdir(parents=True, exist_ok=True)
    responses = []

    with SqliteStore.from_conn_string(
        str(db_path),
        index={
            "embed": memory_embeddings,
            "dims": 384,
            "fields": ["$"],
        },
    ) as store:
        store.setup()

        for sample_index, sample in enumerate(dataset):
            start = perf_counter()
            namespace = ("structured_memory", uuid4().hex, "agent_episodes")

            memory_model = chat.model_copy(update={"num_predict": 4096})

            memory_manager = create_memory_store_manager(
                memory_model,
                schemas=[Episode],
                namespace=namespace,
                store=store,
                query_limit=5,
                enable_inserts=True,
                enable_deletes=False,
                instructions=(
                    "Maintain concise episodic memories about the main character "
                    "and other relevant characters. "
                    "Do not miss important information about the main character. "
                    "Create new episodes or update existing episodes when information changes. "
                    "Represent each episode using who, action, whom, when, and where. "
                    "Keep each field a short, one-line string. "
                    "Preserve important names, titles, dates, and quantities. "
                    "Do not confuse conversation dates with event dates. "
                    "Use empty strings for unknown fields. "
                    "Ignore generic documents and unrelated explanations. "
                    "Example: The user attended The Lord of the Ring at a community theater. "
                    '{"who": "user", "action": "attended", '
                    '"whom": "Lord of the Ring", "when": "", '
                    '"where": "community theater"}'
                ),
            )

            search_memory = create_search_memory_tool(namespace=namespace)

            agent = create_agent(
                model=chat,
                tools=[search_memory],
                store=store,
                system_prompt=(
                    "Answer questions about previous conversations using saved episodes. "
                    "Search memory before answering. "
                    "Preserve the distinction between past and current information. "
                    "Say you do not know when the information is missing."
                ),
            )

            sessions = zip(sample["haystack_dates"], sample["haystack_sessions"], strict=True)
            for date, session in tqdm(
                sessions,
                total=len(sample["haystack_sessions"]),
                desc=f"Sample {sample_index}",
            ):
                for message in session:
                    turn = (
                        f"History turn [{date} | {message['role']}]\n"
                        f"{message['content']}"
                    )
                    memory_manager.invoke({
                        "messages": [HumanMessage(content=turn)],
                        "max_steps": 1,
                    })

            question = sample["question"]
            question_prompt = (
                f"Final question [{sample['question_date']}]\n{question}"
            )
            answer = agent.invoke({"messages": [HumanMessage(content=question_prompt)]})

            memory = []
            while items := store.search(namespace, limit=100, offset=len(memory)):
                memory.extend(item.value["content"] for item in items)

            responses.append({
                "question": question,
                "answer": answer,
                "memory": memory,
                "exec_time": round(perf_counter() - start, 4),
            })

    return responses

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5519.60it/s]

# 6. Graph RAG

In [7]:
graph_embeddings = LangchainEmbedding(embeddings, embed_batch_size=32)

graph_extract_prompt = """Extract up to {max_knowledge_triplets} grounded relations from this dialogue.
Return one line per relation: subject | relation | object. No header or Markdown.

Use the speaker labels to resolve "I". Keep the original spelling of names and titles.
When the speaker first describes something by type and later names it, use the explicit
name as the entity in the action. Link that named entity to its type and location if stated.
If no name is given, do not guess it. Distinguish suggestions from completed actions.
A conversation date is not evidence that an event happened on that date.

Example:
user: I took a class at a studio in Dover. The studio was called Alder Loft.
User | took a class at | Alder Loft
Alder Loft | is a | studio
Alder Loft | located in | Dover

Dialogue:
{text}

Relations:"""

turn_header = re.compile(
    r"(?m)^((?:user|assistant):[ \t]*\[Conversation date:[^\]\n]+\])"
)
graph_excerpt_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    AutoTokenizer.from_pretrained(embedding_name),
    chunk_size=320,
    chunk_overlap=32,
)


def parse_graph_triples(output):
    triples = []
    for line in output.splitlines():
        fields = [field.strip() for field in line.split("|")]
        if len(fields) == 3 and all(fields):
            triple = tuple(fields)
            if triple not in triples:
                triples.append(triple)
    return triples


def build_history_graph(sessions):
    documents = [
        GraphDocument(text="\n".join(f"{turn['role']}: {turn['content']}" for turn in session))
        for session in sessions
        if session
    ]

    llm = Ollama(
        model="qwen3:4b-instruct",
        context_window=32_768,
        temperature=0,
        thinking=False,
        request_timeout=180.0,
        additional_kwargs={"seed": seed, "num_predict": 16_384},
    )
    extractor = SimpleLLMPathExtractor(
        llm=llm,
        extract_prompt=graph_extract_prompt,
        parse_fn=parse_graph_triples,
        max_paths_per_chunk=10,
        num_workers=1,
    )

    return PropertyGraphIndex.from_documents(
        documents,
        llm=llm,
        embed_model=graph_embeddings,
        kg_extractors=[extractor],
        transformations=[SentenceSplitter(chunk_size=1_024, chunk_overlap=32)],
        use_async=False,
        show_progress=True,
    )


def retrieve_graph_evidence(index, question):
    store = index.property_graph_store
    retriever = VectorContextRetriever(
        store,
        vector_store=index.vector_store,
        embed_model=graph_embeddings,
        similarity_top_k=20,
        path_depth=5,
        limit=max(1, 2 * len(store.graph.triplets)),
        include_text=False,
    )

    # Keep the retriever's ordering when removing duplicate paths.
    paths = list(dict.fromkeys(hit.node.get_content() for hit in retriever.retrieve(question)))

    if len(paths) > 512:
        bm25 = BM25Okapi([tokenize(path) for path in paths])
        lexical = bm25.get_top_n(tokenize(question), paths, n=512)
        paths = list(dict.fromkeys(paths[:128] + lexical))[:512]

    ranked = reranker.rank(question, paths, top_k=len(paths)) if paths else []
    sources = {
        f"{rel.source_id} -> {rel.label} -> {rel.target_id}": rel.properties.get(
            "triplet_source_id"
        )
        for rel in store.graph.get_all_relations()
    }

    # Select up to 10 distinct source chunks, keeping their two best relations.
    grouped = {}
    for hit in ranked:
        path = paths[hit["corpus_id"]]
        source_id = sources.get(path)
        if not source_id:
            continue
        if source_id not in grouped:
            if len(grouped) == 10:
                continue
            grouped[source_id] = []
        if len(grouped[source_id]) < 2:
            grouped[source_id].append(path)

    source_nodes = [node for node in store.get() if isinstance(node, ChunkNode)]
    chunks = {node.id: node.text for node in source_nodes}

    evidence = []
    for source_id in chunks:
        # A source chunk can start mid-turn. Keep that text without inventing a speaker.
        parts = turn_header.split(chunks.get(source_id, ""))
        header = ""
        node = index.docstore.get_node(source_id)
        start = node.start_char_idx
        while parts[0].strip() and node.prev_node:
            node = index.docstore.get_node(node.prev_node.node_id)
            headers = [
                match.group(1)
                for match in turn_header.finditer(node.get_content())
                if node.start_char_idx + match.start() < start
            ]
            if headers:
                header = headers[-1]
                break
        turns = [(header, parts[0]), *zip(parts[1::2], parts[2::2])]
        excerpts = []
        for header, body in turns:
            for piece in graph_excerpt_splitter.split_text(body.strip()):
                excerpts.append(f"{header} {piece}".strip())

        excerpts = list(dict.fromkeys(excerpts))
        evidence.extend((source_id, excerpt) for excerpt in excerpts)

    if source_nodes:
        query = np.asarray(graph_embeddings.get_query_embedding(question))
        vectors = np.asarray([node.embedding for node in source_nodes])
        scores = (vectors @ query) / np.maximum(
            np.linalg.norm(vectors, axis=1) * np.linalg.norm(query), 1e-12
        )
        for position in np.argsort(-scores)[:5]:
            grouped.setdefault(source_nodes[position].id, [])
    if evidence:
        bm25 = BM25Okapi([tokenize(text) for _, text in evidence])
        sparse_scores = bm25.get_scores(tokenize(question))
        for position in np.argsort(-sparse_scores)[:5]:
            grouped.setdefault(evidence[position][0], [])

    if not grouped:
        return [] if not paths else [
            "Graph relations:\n"
            + "\n".join(paths[hit["corpus_id"]] for hit in ranked[:10])
        ]

    evidence = [(source_id, text) for source_id, text in evidence if source_id in grouped]

    selected = {}
    ranked = (
        reranker.rank(question, [text for _, text in evidence], top_k=min(20, len(evidence)))
        if evidence else []
    )
    for hit in ranked:
        source_id, excerpt = evidence[hit["corpus_id"]]
        if source_id not in selected:
            if len(selected) == 10:
                continue
            selected[source_id] = []
        selected[source_id].append(excerpt)

    return [
        "Conversation excerpts:\n" + "\n".join(excerpts)
        + (
            "\n\nGraph relations:\n" + "\n".join(grouped[source_id])
            if grouped[source_id] else ""
        )
        for source_id, excerpts in selected.items()
    ]


def chat_with_graph_rag(data):
    agent = create_agent(model=chat, system_prompt=system_prompt)
    responses = []

    with ThreadPoolExecutor(max_workers=1) as worker:
        for sample in tqdm(data, desc="Graph RAG"):
            start = perf_counter()
            index = worker.submit(build_history_graph, dated_sessions(sample)).result()
            indexed = perf_counter()

            question = sample["question"]
            question_text = dated_question(sample).content
            evidence = retrieve_graph_evidence(index, question)

            # Include instructions and the dated question in the input estimate.
            input_budget = chat.num_ctx - chat.num_predict - 512
            while True:
                context = "\n\n".join(evidence) or "(No graph evidence found.)"
                prompt = HumanMessage(content=(
                    "Answer from the graph evidence. Check names, speakers and dates "
                    "against the quoted dialogue when available.\n\n"
                    f"{context}\n\n{question_text}"
                ))
                if count_tokens_approximately([system_prompt, prompt]) <= input_budget:
                    break
                if not evidence:
                    raise ValueError("The question exceeds the answer model's input budget.")
                evidence.pop()

            answer = agent.invoke({"messages": [prompt]})
            end = perf_counter()

            responses.append({
                "question": question,
                "answer": answer,
                "retrieved": evidence,
                "graph_store": index.property_graph_store,
                "index_time": round(indexed - start, 4),
                "query_time": round(end - indexed, 4),
                "exec_time": round(end - start, 4),
            })

    return responses

In [7]:
responses = chat_with_graph_rag(dataset[:1])

for r in responses:
    print("# 6. Graph RAG")
    print(f"[Question] {r['question']}")
    print(f"[Answer] {r['answer']['messages'][-1].content}")
    print(f"[Indexing] {r['index_time']:.2f}s")
    print(f"[Retrieval + generation] {r['query_time']:.2f}s")
    print(f"[Total] {r['exec_time']:.2f}s")

    for rank, evidence in enumerate(r["retrieved"], 1):
        print(f"\n[Graph evidence {rank}]\n{evidence}")

Graph RAG:   0%|          | 0/1 [00:00<?, ?it/s]

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Applying transformations: 100%|██████████| 1/1 [00:00<00:00,  1.11it/s]

Applying transformations: 100%|██████████| 1/1 [00:00<00:00,  1.11it/s]

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Applying transformations: 100%|██████████| 1/1 [01:35<00:00, 95.71s/it]

... (132 outputs left out)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (514 > 512). Running this sequence through the model will result in indexing errors


Graph RAG: 100%|██████████| 1/1 [02:11<00:00, 131.86s/it]

Graph RAG: 100%|██████████| 1/1 [02:11<00:00, 131.86s/it]

# 6. Graph RAG
[Question] What degree did I graduate with?
[Answer] You graduated with a degree in Business Administration.
[Indexing] 125.55s
[Retrieval + generation] 6.32s
[Total] 131.86s

[Graph evidence 1]
Conversation excerpts:
user: [Conversation date: 2023/05/30 (Tue) 17:27] I graduated with a degree in Business Administration, which has definitely helped me in my new role. Do you have any advice on how to stay organized when it comes to paperwork and documentation, especially when it comes to tracking expenses and reimbursements?
assistant: [Conversation date: 2023/05/30 (Tue) 17:27] Congratulations on your degree in Business Administration! That's a great foundation for your new role.

Staying organized with paperwork and documentation is crucial, especially when it comes to tracking expenses and reimbursements. Here are some tips to help you stay on top of it:

[Graph evidence 2]
Conversation excerpts:
assistant: [Conversation date: 2023/05/29 (Mon) 19:30] To the esteemed Hir

In [8]:
from utils import export_memory_graphs

# Graphical visualization
full_path, retrieved_path = export_memory_graphs(responses[0], output_dir="output/graph")

print(full_path.resolve())
print(retrieved_path.resolve())

C:\Users\yana\Desktop\Library\AI\ai-topic\ai-topic-memory\output\graph\graph_full.html
C:\Users\yana\Desktop\Library\AI\ai-topic\ai-topic-memory\output\graph\graph_retrieved.html


# Evaluation

## Sanity Check

In [13]:
class Grade(BaseModel):
    correct: bool = Field(strict=True)


judge = chat.model_copy(update={"temperature": 0})


judge_prompt = """
Compare the CANDIDATE's answer to the QUESTION with the REFERENCE.
Apply the question-type rule below, if provided.

Mark true if the candidate states the required answer, clearly implies it,
or gives all the facts needed to derive it. Accept equivalent wording.
A caveat about independent verification is fine if the candidate still gives the answer.
If the candidate can be reasonably estimated to know the reference, mark true.

For an answerable question, mark false if the candidate concludes that the answer
is unknown or gives a conflicting final answer. For a multi-part question,
each value must answer the right part, for the right person, quantity, and time.
""".strip()


type_rules = {
    "knowledge-update": (
        "Distinguish previous and current values. If both are requested, "
        "the candidate must give both in the correct order."
    ),
    "temporal-reasoning": (
        "For a duration count, accept an off-by-one difference in the requested "
        "unit. Calendar dates and clock times must match."
    ),
    "single-session-preference": (
        "The reference describes the user's preference. The candidate must use "
        "that preference in its advice. Generic advice alone is incorrect."
    ),
    "abstention": (
        "This question is unanswerable. Mark true if the candidate recognizes "
        "that the requested information is missing and gives no definite guess."
    ),
}


def grade_answer(case):
    rule = type_rules.get(case["type"], "")
    comparison = {
        "question": case["question"],
        "reference": case["reference"],
        "candidate": case["candidate"],
    }
    answer = judge.invoke([
        ("system", f"{judge_prompt}\n{rule}\nReturn only a JSON object with one boolean field correct."),
        ("human", json.dumps(comparison, ensure_ascii=False)),
    ])
    return Grade.model_validate_json(answer.content)


# Compare expected labels in Python only.
controls = [
    (
        "single-session-user", "Where does my sister Emily live?", "Denver",
        "There is no information about where Emily lives. Her location cannot be determined.",
        False,
    ),
    (
        "single-session-user", "Where does my sister Emily live?", "Denver",
        "Your sister Emily lives in Denver.",
        True,
    ),
    (
        "single-session-user", "How many playlists do I have on Spotify?", "20",
        'You said you have "20 playlists on Spotify already." '
        "I cannot independently verify your account.",
        True,
    ),
    (
        "single-session-user", "How many playlists do I have on Spotify?", "20",
        "You mentioned 20 songs. The number of playlists is unknown.",
        False,
    ),
    (
        "knowledge-update", "How many engineers did I lead initially and how many now?",
        "Initially 4 engineers, now 5 engineers.",
        "You led 4 engineers. Including your manager, 5 people attended the outing.",
        False,
    ),
    (
        "knowledge-update", "How many engineers did I lead initially and how many now?",
        "Initially 4 engineers, now 5 engineers.",
        "You started with four engineers and now lead five.",
        True,
    ),
]

JUDGE_OK = False
failures = []

for kind, question, reference, candidate, expected in controls:
    case = {
        "type": kind,
        "question": question,
        "reference": reference,
        "candidate": candidate,
    }
    actual = grade_answer(case).correct
    if actual != expected:
        failures.append({**case, "expected": expected, "actual": actual})

JUDGE_OK = not failures
print(f"Judge controls: {len(controls) - len(failures)}/{len(controls)}")

if failures:
    display(pd.DataFrame(failures))
    raise RuntimeError("Judge controls failed. Inspect the failures before evaluation.")

Judge controls: 6/6


## Run

In [14]:
samples = random.Random(52).sample(dataset, 2)
methods = {
    "Graph RAG": chat_with_graph_rag,
}

rows = []
for name, run in methods.items():
    for s in tqdm(samples, desc=name):
        start = perf_counter()
        result = run([s])[0]
        answer = result["answer"]
        seconds = perf_counter() - start
        prediction = (answer["messages"][-1] if isinstance(answer, dict) else answer).content

        case = {
            "type": "abstention" if s["question_id"].endswith("_abs") else s["question_type"],
            "question": s["question"],
            "reference": s["answer"],
            "candidate": prediction,
        }
        grade = grade_answer(case)
        rows.append({
            **case,
            "id": s["question_id"],
            "method": name,
            "correct": grade.correct,
            "seconds": seconds,
            "retrieved": json.dumps(result.get("retrieved", []), ensure_ascii=False),
        })
        print(f"Question: {s['question']}")
        print(f"Correct: {grade.correct}")
        print(f"Reference: {rows[-1]['reference']}")
        print(f"Candidate: {rows[-1]['candidate']}")
        pprint(f"Retrieved:\n{rows[-1]['retrieved']}")

details = pd.DataFrame(rows)
details.to_csv("evaluation_results.csv", index=False, encoding="utf-8-sig")

display(details.groupby("method", sort=False).agg(
    n=("correct", "size"),
    correct=("correct", "sum"),
    accuracy_pct=("correct", lambda x: 100 * x.mean()),
    mean_seconds=("seconds", "mean"),
).round(2))

Graph RAG:   0%|          | 0/2 [00:00<?, ?it/s]

Graph RAG:   0%|          | 0/1 [00:00<?, ?it/s]

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Applying transformations: 100%|██████████| 1/1 [00:00<00:00,  1.55it/s]

Applying transformations: 100%|██████████| 1/1 [00:00<00:00,  1.55it/s]

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

... (282 outputs left out)

Graph RAG: 100%|██████████| 2/2 [04:45<00:00, 142.30s/it]

Graph RAG: 100%|██████████| 2/2 [04:45<00:00, 142.57s/it]

Question: How old was I when my grandma gave me the silver necklace?
Correct: True
Reference: 18
Candidate: The user was 18 years old when their grandma gave them the silver necklace.
('Retrieved:\n'
 '["Conversation excerpts:\\nuser: [Conversation date: 2023/05/22 (Mon) 16:18] '
 "I'm going to start by adding the silver necklace my grandma gave me on my "
 '18th birthday. I\'ll write in the \\"Provenance/Story\\" column that it was '
 "a gift from her, and that I lost it about two months ago at my friend's "
 "house during a sleepover. I'm still hoping to find it, but in the meantime, "
 "I'm making a note to ask my friend if she's seen it.\\nuser: [Conversation "
 'date: 2023/05/22 (Mon) 16:18] I like the idea of a \\"Provenance/Story\\" '
 "column. It'll be nice to have a record of the stories behind each piece. I "
 "think I'll start by adding the pieces that are most sentimental to me, like "
 'the silver necklace my grandma gave me on my 18th birthday. I remember I '
 "lost it a 

,n,correct,accuracy_pct,mean_seconds
method,,,,
Graph RAG,2,1,50.0,141.64


In [15]:
details_graph = details[details['method'] == 'Graph RAG']

for idx, row in details_graph.iterrows():
    if row['correct'] == False:
        print(row['question'])
        print(row['reference'])
        print(row['candidate'])
        print(row['correct'])
        print(len(row['retrieved']))
        pprint(row['retrieved'])
        print("\n\n")

Can you recommend a show or movie for me to watch tonight?
The user would prefer recommendations for stand-up comedy specials on Netflix, especially those that are known for their storytelling. They may not prefer recommendations for other genres or platforms.
The BBC documentary "Ancient Egypt: The Greatest Show on Earth" (2003) is recommended for its in-depth exploration of the topic.
False
7469
('["Conversation excerpts:\\nuser: [Conversation date: 2023/05/29 (Mon) 04:39] '
 "As an aspiring stand-up comedian, I'm looking for some advice on how to "
 'improve my craft. Can you recommend some stand-up comedy specials on Netflix '
 "with strong storytelling abilities like John Mulaney's 'Kid "
 "Gorgeous'?\\nassistant: [Conversation date: 2023/05/29 (Mon) 04:39] John "
 'Mulaney\'s \\"Kid Gorgeous\\" is an excellent example of standout '
 'storytelling in stand-up comedy. Here are some Netflix stand-up specials '
 'with strong storytelling abilities that you might enjoy and learn '
 'f

Reference: The user would prefer responses that suggest healthy meal prep recipes, especially those that incorporate quinoa and roasted vegetables, and offer variations in protein sources. They might appreciate suggestions that build upon their existing preferences, such as new twists on chicken Caesar salads or turkey and avocado wraps. The user may not prefer responses that suggest unhealthy or high-calorie meal prep options, or those that deviate significantly from their established healthy eating habits.
Candidate: I recommend seasoning your roasted sweet potatoes with rosemary, thyme, and sage, as they pair well together and enhance the dish's flavor.